# Delivery Time Prediction — Data Cleaning

**Steps (exactly):**
1. Drop specified columns
2. Drop rows with missing values
3. Clip target at p99
4. Log-transform heavy-tailed numeric features

**Output:** `train_clean.parquet` / `test_clean.parquet`

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TARGET = 'delivery_time_days'

DROP_COLS = [ 
    'id',
    'order_id', 'customer_unique_id', 'product_id',
    'product_name_length', 'product_description_length',
    'product_length_cm', 'product_height_cm', 'product_width_cm', 
    'product_photos_qty'
]

LOG_COLS = [
    'price', 'freight_value', 'product_weight_g',
    'volume_cm3', 'quantity',
]

print('Setup complete ✓')

---
## 1. Load data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')

---
## 2. Drop columns

In [ ]:
train = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
test  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')

---
## 3. Drop rows with missing values

In [ ]:
n_train_before = len(train)
n_test_before  = len(test)

train = train.dropna().reset_index(drop=True)
test  = test.dropna().reset_index(drop=True)

print(f'Train : {n_train_before:,} → {len(train):,}  (dropped {n_train_before - len(train):,})')
print(f'Test  : {n_test_before:,}  → {len(test):,}   (dropped {n_test_before  - len(test):,})')

assert train.isnull().sum().sum() == 0, 'Missing values remain in train!'
assert test.isnull().sum().sum()  == 0, 'Missing values remain in test!'
print('✓ No missing values remain')

---
## 4. Clip target at p99

In [ ]:
p99 = train[TARGET].quantile(0.99)
n_clipped = (train[TARGET] > p99).sum()

train[TARGET] = train[TARGET].clip(upper=p99)

print(f'p99 threshold : {p99:.2f} days')
print(f'Rows clipped  : {n_clipped} ({n_clipped / len(train) * 100:.2f}%)')
print(f'New max       : {train[TARGET].max():.2f} days')

---
## 5. Log-transform heavy-tailed numeric features

In [ ]:
log_cols_present = [c for c in LOG_COLS if c in train.columns]

for col in log_cols_present:
    for df in (train, test):
        df[col] = np.log1p(df[col].clip(lower=0))

print(f'log1p applied to: {log_cols_present}')

---
## 6. Validation

In [ ]:
assert train.isnull().sum().sum() == 0,           'Missing values in train!'
assert test.isnull().sum().sum()  == 0,           'Missing values in test!'
assert train[TARGET].max() <= p99,                'Target not clipped!'
assert np.isinf(train.select_dtypes(include=np.number).values).sum() == 0, 'Inf values in train!'
assert np.isinf(test.select_dtypes(include=np.number).values).sum()  == 0, 'Inf values in test!'
assert TARGET not in test.columns,                'Target leaking into test!'

print('All checks passed ✓')
print(f'\nFinal shapes — Train: {train.shape}  |  Test: {test.shape}')
print(f'Columns: {list(train.columns)}')

---
## 7. Save

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

train.to_parquet('../data/processed/train_clean.parquet', index=False)
test.to_parquet( '../data/processed/test_clean.parquet',  index=False)

print('Saved train_clean.parquet ✓')
print('Saved test_clean.parquet  ✓')